In [1]:
import json
import pandas as pd
from pathlib import Path

# Define the base results directory (relative to notebook location)
results_base_dir = "../results"

# Load webp variant glossary results
print("📊 Loading WEBP Glossary Results")
print("=" * 60)

webp_dir = Path(results_base_dir) / "gemini-2.5-flash-webp-dhao"

# Load both glossary configurations
glossary_configs = {
    "glossary": "Fuzzy Word",
    "glossary_full": "Full"
}

webp_data = {}
for config_dir, method_name in glossary_configs.items():
    config_path = webp_dir / config_dir
    metrics_file = list(config_path.glob("*_metrics.json"))[0]
    
    with open(metrics_file, 'r') as f:
        webp_data[method_name] = json.load(f)
    
    print(f"✅ Loaded {method_name}: {metrics_file.name}")

print("=" * 60)


📊 Loading WEBP Glossary Results
✅ Loaded Fuzzy Word: gemini-2.5-flash_aligned-eng-engwebp-ot_eng_nfa_metrics.json
✅ Loaded Full: gemini-2.5-flash_aligned-eng-engwebp-ot_eng_nfa_metrics.json


In [12]:
# Create DataFrame with absolute scores and improvements
rows = []

# Get baseline scores from the first method (both have same baseline)
first_method = list(webp_data.values())[0]
baseline_row = {
    'Method': 'nllb-200-distilled-600M_NT',
    'n_shot': "~",
    'spBLEU': f"{first_method['original_mt_metrics']['spbleu']:.2f}",
    'chrF++': f"{first_method['original_mt_metrics']['chrf++']:.2f}"
}
rows.append(baseline_row)

# Define n_shot values for each method
n_shot_values = {
    "Fuzzy Word": "5 (≈127)",
    "Full": 2377
}

# Add rows for each post-editing method with improvements
for method_name, metrics in webp_data.items():
    spbleu_score = metrics['post_edited_metrics']['spbleu']
    spbleu_improvement = metrics['improvements']['spbleu']
    chrf_score = metrics['post_edited_metrics']['chrf++']
    chrf_improvement = metrics['improvements']['chrf++']
    
    row = {
        'Method': method_name,
        'n_shot': n_shot_values[method_name],
        'spBLEU': f"{spbleu_score:.2f} (+{spbleu_improvement:.2f})",
        'chrF++': f"{chrf_score:.2f} (+{chrf_improvement:.2f})"
    }
    rows.append(row)

# Create DataFrame
df_webp = pd.DataFrame(rows)

# Display the results
print("\n📋 WEBP Variant Glossary Post-Editing Results")
print("=" * 80)
df_webp


📋 WEBP Variant Glossary Post-Editing Results


,Method,n_shot,spBLEU,chrF++
0,nllb-200-distilled-600M_NT,~,7.66,27.11
1,Fuzzy Word,5 (≈127),14.29 (+6.63),29.76 (+2.65)
2,Full,2377,16.27 (+8.61),31.32 (+4.21)
